# AI Data Chatbot
Ask natural language questions about your dataset — get back tables and interactive charts.

**Run all cells top to bottom, then use the chat widget at the bottom.**

In [ ]:
# ── Cell 1: Install dependencies (run once) ──────────────────────────────────
# Uncomment and run this cell if you haven't installed the packages yet
# !pip install anthropic pandas numpy plotly openpyxl python-dotenv ipywidgets

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os, json, re, textwrap
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env file
print('✓ Imports OK')

In [ ]:
# ── Cell 3: API Key ───────────────────────────────────────────────────────────
# Option A: reads from .env file automatically (recommended)
# Option B: paste your key directly below
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

assert os.getenv('ANTHROPIC_API_KEY'), "Set ANTHROPIC_API_KEY in .env or uncomment Option B above"
print('✓ API key found')

In [ ]:
# ── Cell 4: Load Data ─────────────────────────────────────────────────────────
# Default: load the sample employee dataset
# To use your own file, change this path to any CSV or Excel file:
DATA_PATH = 'sample_data/employees.csv'

if DATA_PATH.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f'✓ Loaded {len(df):,} rows × {len(df.columns)} columns')
print(f'  Columns: {", ".join(df.columns.tolist())}')
df.head()

In [ ]:
# ── Cell 5: Core helpers ──────────────────────────────────────────────────────

def get_schema_info(df):
    lines = [f'Shape: {df.shape[0]} rows × {df.shape[1]} columns', 'Columns:']
    for col in df.columns:
        dtype = str(df[col].dtype)
        if pd.api.types.is_numeric_dtype(df[col]):
            lines.append(f'  {col} ({dtype}): min={df[col].min()}, max={df[col].max()}, mean={df[col].mean():.1f}')
        else:
            uniq = df[col].nunique()
            sample = df[col].dropna().unique()[:5].tolist()
            lines.append(f'  {col} ({dtype}): {uniq} unique values, e.g. {sample}')
    return '\n'.join(lines)

def strip_imports(code):
    return '\n'.join(
        line for line in code.splitlines()
        if not line.strip().startswith(('import ', 'from '))
    )

def safe_exec_pandas(code, df):
    code = strip_imports(code)
    ns = {'df': df.copy(), 'pd': pd, 'np': np,
          '__builtins__': {'len':len,'range':range,'print':print,'str':str,
                           'int':int,'float':float,'list':list,'dict':dict,
                           'zip':zip,'enumerate':enumerate,'sorted':sorted,
                           'min':min,'max':max,'sum':sum,'abs':abs,'round':round}}
    exec(code, ns)
    return ns.get('result_df')

def safe_exec_viz(code, result_df):
    code = strip_imports(code)
    ns = {'result_df': result_df.copy(), 'px': px, 'go': go, 'pd': pd, 'np': np,
          '__builtins__': {'len':len,'range':range,'print':print,'str':str,
                           'int':int,'float':float,'list':list,'dict':dict,
                           'zip':zip,'enumerate':enumerate,'sorted':sorted,
                           'min':min,'max':max,'sum':sum,'abs':abs,'round':round}}
    exec(code, ns)
    return ns.get('fig')

def fallback_chart(result_df):
    num_cols = result_df.select_dtypes(include='number').columns.tolist()
    cat_cols = result_df.select_dtypes(exclude='number').columns.tolist()
    if num_cols and cat_cols:
        return px.bar(result_df, x=cat_cols[0], y=num_cols[0], title='Results')
    elif num_cols:
        return px.bar(result_df, y=num_cols[0], title='Results')
    return None

print('✓ Helpers ready')

In [ ]:
# ── Cell 6: LLM handler ───────────────────────────────────────────────────────

SYSTEM_PROMPT = """
You are a data analyst assistant. The user will ask questions about a dataset.
You must respond with ONLY a valid JSON object — no markdown, no explanation outside the JSON.

JSON format:
{
  "explanation": "Plain-English summary of what you found",
  "pandas_code": "Python code using pandas. Input: `df`. Output: must assign a DataFrame to `result_df`.",
  "viz_code": "Python code using plotly. Input: `result_df`. Output: must assign a Plotly figure to `fig`. DO NOT include any import statements.",
  "viz_type": "bar | line | pie | scatter | heatmap | table_only | null"
}

Rules:
- pandas_code must always produce a `result_df` DataFrame
- viz_code must always produce a `fig` Plotly figure (never use import statements)
- If no chart makes sense, set viz_type to table_only and viz_code to empty string
"""

client = anthropic.Anthropic()
conversation_history = []
schema_info = get_schema_info(df)
sample_rows = df.head(5).to_string()

def ask(question):
    user_msg = f"""Dataset schema:\n{schema_info}\n\nSample rows:\n{sample_rows}\n\nQuestion: {question}"""
    conversation_history.append({'role': 'user', 'content': user_msg})

    response = client.messages.create(
        model='claude-opus-4-6',
        max_tokens=4096,
        thinking={'type': 'adaptive'},
        system=SYSTEM_PROMPT,
        messages=conversation_history,
    )

    raw = next((b.text for b in response.content if hasattr(b, 'text')), '')
    conversation_history.append({'role': 'assistant', 'content': raw})

    # Parse JSON
    text = re.sub(r'^```[\w]*\n?', '', raw.strip())
    text = re.sub(r'\n?```$', '', text.strip())
    m = re.search(r'\{[\s\S]*\}', text)
    return json.loads(m.group(0) if m else text)

print('✓ LLM handler ready')

In [ ]:
# ── Cell 7: Interactive Chat Widget ───────────────────────────────────────────

chat_log = widgets.Output()
text_input = widgets.Text(
    placeholder='Ask a question about your data...',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(description='Ask', button_style='primary',
                          layout=widgets.Layout(width='10%'))
clear_btn = widgets.Button(description='Clear chat', button_style='warning',
                           layout=widgets.Layout(width='12%'))

EXAMPLES = [
    'Compare 2022 vs 2023 highest paid employees by job title',
    'Top 10 highest paid employees in 2023',
    'Average salary by department for each year',
    'Which department has highest salary growth 2021 to 2024?',
    'Show salary distribution by location in 2023',
]
example_btns = [widgets.Button(description=e[:55], layout=widgets.Layout(width='100%'))
                for e in EXAMPLES]

def run_query(question):
    with chat_log:
        print('\n' + '-' * 60)
        display(HTML('<b style="color:#1a73e8">You:</b> ' + question))
        display(HTML('<i style="color:gray">Thinking...</i>'))

    try:
        result = ask(question)
    except Exception as e:
        with chat_log:
            clear_output(wait=True)
            display(HTML('<b style="color:red">Error calling LLM:</b> ' + str(e)))
        return

    with chat_log:
        clear_output(wait=True)
        display(HTML('<b style="color:#1a73e8">You:</b> ' + question))
        display(HTML('<b>Assistant:</b> ' + result.get('explanation', '') + '<br>'))

        # Run pandas code
        pandas_code = result.get('pandas_code', '')
        try:
            result_df = safe_exec_pandas(pandas_code, df)
            if result_df is not None and not result_df.empty:
                display(HTML('<b>Data Table:</b>'))
                display(result_df)
            else:
                display(HTML('<i style="color:orange">No data returned.</i>'))
                return
        except Exception as e:
            display(HTML('<b style="color:red">Pandas error:</b> ' + str(e)))
            display(HTML('<pre style="font-size:0.8em;color:gray">' + pandas_code + '</pre>'))
            return

        # Run viz code
        viz_type = result.get('viz_type', '')
        viz_code = result.get('viz_code', '')
        if viz_type != 'table_only' and viz_code:
            try:
                fig = safe_exec_viz(viz_code, result_df)
                if fig:
                    fig.show()
                else:
                    raise ValueError('fig is None')
            except Exception:
                fig = fallback_chart(result_df)
                if fig:
                    fig.show()

def on_send(b):
    q = text_input.value.strip()
    if q:
        text_input.value = ''
        run_query(q)

def on_example(b):
    run_query(b.description)

def on_clear(b):
    global conversation_history
    conversation_history = []
    with chat_log:
        clear_output()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)
for btn in example_btns:
    btn.on_click(on_example)

display(HTML('<h3>AI Data Chatbot</h3><p>Type a question or click an example below:</p>'))
display(widgets.VBox(example_btns))
display(HTML('<br>'))
display(widgets.HBox([text_input, send_btn, clear_btn]))
display(chat_log)